# MÓDULO 3
## Tema 4. Composición, herencia y polimorfismo: ejemplos prácticos

**Objetivos:**
- Diferenciar con claridad **composición** vs **herencia**.
- Aplicar **herencia** (simple, multinivel y múltiple) entendiendo sus implicaciones.
- Entender y aplicar **polimorfismo** (mismo mensaje, distinto comportamiento) en Python.

## Índice
- Composición ("tiene un")
- Herencia simple ("es un")
- Herencia multinivel
- Herencia múltiple (varios padres) y sus problemas
- Polimorfismo (duck typing)
- Trampas comunes y buenas prácticas

## Composición ("tiene un")

La **composición** modela relaciones del tipo:

- Un `Coche` **tiene un** `Motor`
- Un `Pedido` **tiene** `Productos` 

Ventajas típicas:
- Menos acoplamiento que una jerarquía de herencia
- Más flexibilidad (puedes cambiar piezas sin cambiar el tipo principal)
- Mejora la testabilidad (puedes inyectar dependencias)

En la práctica, composición suele ser el “default”.

### Ejemplo realista: un coche tiene un motor

La clase `Coche` delega parte del comportamiento en `Motor`.

In [ ]:
class Motor:
    def __init__(self, potencia_cv: int):
        self.potencia_cv = int(potencia_cv)
        self.encendido = False

    def encender(self):
        self.encendido = True

    def apagar(self):
        self.encendido = False

    def __str__(self):
        return f"Motor({self.potencia_cv}cv, encendido={self.encendido})"


class Coche:
    def __init__(self, marca: str, motor: Motor):
        self.marca = marca
        self.motor = motor  # composición: "tiene un"
        print(f"Coche creado: {self.marca} con {self.motor}")

    def arrancar(self):
        self.motor.encender()
        return f"{self.marca}: {self.motor}"

    def parar(self):
        self.motor.apagar()
        return f"{self.marca}: {self.motor}"


m1 = Motor(potencia_cv=110)
c1 = Coche(marca="Seat", motor=m1)
print(c1.marca)
print(c1.motor)
print(c1.motor.potencia_cv)
print(c1.motor.encendido)
print(f"Lucia tiene un {c1.marca} con un motor de {c1.motor.potencia_cv}cv")

Coche creado: Seat con Motor(110cv, encendido=False)
Seat
Motor(110cv, encendido=False)
110
False
Lucia tiene un Seat con un motor de 110cv


In [11]:
print(c1.arrancar())
print(c1.parar())

Seat: Motor(110cv, encendido=True)
Seat: Motor(110cv, encendido=False)


### Ejemplo realista: un pedido tiene productos

In [ ]:
# Opción 1
class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

class Pedido:
    def __init__(self):
        self.productos = []

    def añadir_producto(self, producto):
        self.productos.append(producto)

    def total(self):
        # Cálculo paso a paso, más explícito
        total = 0
        for producto in self.productos:
            total += producto.precio
        return total

# Uso
p1 = Producto("Camiseta", 20.0)
p2 = Producto("Pantalón", 35.0)

pedido = Pedido()
pedido.añadir_producto(p1)
pedido.añadir_producto(p2)

print("Total del pedido:", pedido.total(), "€")

Total del pedido: 55.0 €


In [ ]:
# Opción 2
class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

class Pedido:
    # No usar nunca [], {} o set() como valor por defecto porque son mutables
    # y se comparten entre todas las llamadas (se crean una sola vez)
    def __init__(self, productos=None):
        if productos is None:
            self.productos = []
        else:
            self.productos = productos

    def añadir_producto(self, producto):
        self.productos.append(producto)

    def total(self):
        # Cálculo paso a paso, más explícito
        total = 0
        for producto in self.productos:
            total += producto.precio
        return total

# Uso
p1 = Producto("Camiseta", 20.0)
p2 = Producto("Pantalón", 35.0)

pedido = Pedido([p1, p2])

print("Total del pedido:", pedido.total(), "€")

### Comparación

| Aspecto        | Opción 1                          | Opción 2                          |
|----------------|----------------------------------|----------------------------------|
| Simplicidad    | Muy simple                       | Algo más compleja                |
| Flexibilidad   | Baja (añadir después)            | Alta (inicializa con lista)      |
| Uso            | Más líneas                       | Más compacto                     |
| Aprendizaje    | Mejor para empezar               | Requiere entender `None`         |
| Buenas prácticas | Correcta                      | Más profesional/reutilizable     |

**Conclusión:**  
- Opción 1 → didáctica  
- Opción 2 → más flexible y usada en producción

## Herencia simple ("es un")

La **herencia** modela relaciones del tipo “**es un**”:

- Un `Coche` **es un** `Vehiculo`
- Un `Perro` **es un** `Animal`

Ventajas:
- Reutilización y extensión de código (cuando la jerarquía es natural)

Riesgos:
- Acoplamiento: cambios en la clase base pueden romper clases hijas
- Jerarquías profundas tienden a complicar el diseño

Usa herencia cuando la relación sea clara y estable.

### Ejemplo: `Vehiculo` → `Coche`

In [ ]:
class Vehiculo:
    def __init__(self, marca):
        self.marca = marca

    def arrancar(self):
        return f"{self.marca}: arrancando"

    def detener(self):
        return f"{self.marca}: detenido"


class Coche2(Vehiculo):
    def __init__(self, marca, puertas):
        super().__init__(marca)
        self.puertas = int(puertas)

    def descripcion(self):
        return f"Coche(marca={self.marca}, puertas={self.puertas})"

v = Vehiculo(marca="Genérico")
c2 = Coche2(marca="Toyota", puertas=5)
print(v.arrancar())
print(c2.arrancar())
print(c2.descripcion())

Genérico: arrancando
Toyota: arrancando
Coche(marca=Toyota, puertas=5)


## Herencia multinivel

Herencia multinivel = una cadena de herencia:

`A` → `B` → `C`

Ejemplo realista:
- `Empleado` (base)
- `EmpleadoTecnico` (especialización)
- `EmpleadoBackend` (especialización aún más concreta)

Esto puede ser útil, pero si empieza a crecer mucho, suele ser señal de que la jerarquía se está forzando.

In [6]:
class Empleado:
    def __init__(self, nombre):
        self.nombre = nombre

    def rol(self):
        return "empleado"


class EmpleadoTecnico(Empleado):
    def __init__(self, nombre, stack):
        super().__init__(nombre)
        self.stack = stack

    def rol(self):
        return "técnico"


class EmpleadoBackend(EmpleadoTecnico):
    def __init__(self, nombre, stack, lenguaje):
        super().__init__(nombre, stack)
        self.lenguaje = lenguaje

    def rol(self):
        return "backend"


e = EmpleadoBackend(nombre="Marta", stack="web", lenguaje="Python")
print(e.nombre, e.rol(), e.stack, e.lenguaje)

Marta backend web Python


## Herencia múltiple (varios padres) y sus problemas

Herencia múltiple = una clase hereda de **más de una** clase base:

`class Hija(Padre1, Padre2): ...`

Python lo soporta, pero trae complejidad:

- **Conflictos de nombres** (dos padres definen el mismo método/atributo)
- **Orden de resolución de métodos (MRO)**: ¿a qué método llama realmente?
- **Problemas tipo “diamante”**: dos ramas heredan de una misma base común
- Uso de `super()` en herencia múltiple requiere estilo *cooperativo* para no llamar dos veces a la misma base

Por todo esto, **no suele recomendarse** como primera herramienta de diseño.
En la práctica, si necesitas “combinar comportamientos”, a menudo es mejor:
- composición
- o clases muy pequeñas y controladas (caso especial)

### El orden importa: MRO (Method Resolution Order)

Python decide qué implementación usar siguiendo un orden lineal llamado MRO.

In [7]:
class A:
    def quien(self):
        return "A"

class B(A):
    def quien(self):
        return "B"

class C(A):
    def quien(self):
        return "C"

class D(B, C):
    pass

d = D()
print("d.quien():", d.quien())
print("MRO:", [cls.__name__ for cls in D.__mro__])

d.quien(): B
MRO: ['D', 'B', 'C', 'A', 'object']


### El “problema del diamante” (idea)

      A
     / \
    B   C
     \ /
      D

No siempre “explota”, pero puede generar confusión si no sabes exactamente qué estás haciendo.

Si vas a usar herencia múltiple:
- que sea con clases pequeñas
- evita estado complejo en múltiples padres
- entiende bien MRO y `super()`

Si no, preferible: composición.

In [8]:
# PROBLEMA DEL DIAMANTE (temática RPG)
#        Personaje
#        /       \
#   Guerrero     Mago
#        \       /
#        Paladin

class Personaje:
    def atacar(self):
        print("Personaje.atacar -> golpe básico")


class Guerrero(Personaje):
    def atacar(self):
        print("Guerrero.atacar -> +espada")
        super().atacar()  # pasa el turno al siguiente en el MRO


class Mago(Personaje):
    def atacar(self):
        print("Mago.atacar -> +hechizo")
        super().atacar()  # pasa el turno al siguiente en el MRO


class Paladin(Guerrero, Mago):
    def atacar(self):
        print("Paladin.atacar -> aura sagrada")
        super().atacar()  # sigue la cadena cooperativa


p = Paladin()

print("MRO:", [cls.__name__ for cls in Paladin.mro()])
print("\n--- ataque del paladín ---")
p.atacar()


MRO: ['Paladin', 'Guerrero', 'Mago', 'Personaje', 'object']

--- ataque del paladín ---
Paladin.atacar -> aura sagrada
Guerrero.atacar -> +espada
Mago.atacar -> +hechizo
Personaje.atacar -> golpe básico


In [9]:
# Ahora el “problema” de verdad: si NO usas super() (duplicas la base)
# Aquí está el fallo clásico: llamar a Personaje.atacar(self) a mano.

class Guerrero_mal(Personaje):
    def atacar(self):
        print("Guerrero_mal.atacar -> +espada")
        Personaje.atacar(self)  # ❌ rompe el modelo cooperativo

class Mago_mal(Personaje):
    def atacar(self):
        print("Mago_mal.atacar -> +hechizo")
        Personaje.atacar(self)  # ❌ rompe el modelo cooperativo

class Paladin_mal(Guerrero_mal, Mago_mal):
    def atacar(self):
        print("Paladin_mal.atacar -> aura sagrada")
        Guerrero_mal.atacar(self)  # ❌ llamadas manuales
        Mago_mal.atacar(self)      # ❌ llamadas manuales


p2 = Paladin_mal()

print("\n--- ataque del paladín MAL ---")
p2.atacar()



--- ataque del paladín MAL ---
Paladin_mal.atacar -> aura sagrada
Guerrero_mal.atacar -> +espada
Personaje.atacar -> golpe básico
Mago_mal.atacar -> +hechizo
Personaje.atacar -> golpe básico


## Polimorfismo (duck typing)

Polimorfismo: **mismo mensaje, distinto comportamiento**.

En Python, muchas veces no necesitas herencia explícita.
Si un objeto tiene el método esperado, “vale” (duck typing).

In [10]:
class ReproductorMP3:
    def reproducir(self, archivo):
        print(f"Reproduciendo MP3: {archivo}")


class ReproductorVideo:
    def reproducir(self, archivo):
        print(f"Reproduciendo vídeo: {archivo}")


class ReproductorRadio:
    def reproducir(self, archivo):
        print(f"Sintonizando emisora: {archivo}")


def reproducir_todo(reproductores, archivo):
    for r in reproductores:
        r.reproducir(archivo)

reproducir_todo(
    [ReproductorMP3(), ReproductorVideo(), ReproductorRadio()],
    "stream_01"
)


Reproduciendo MP3: stream_01
Reproduciendo vídeo: stream_01
Sintonizando emisora: stream_01


### Polimorfismo con herencia (también válido)

Aquí sí usamos una base común `Animal`, pero lo importante es que todos responden a `hablar()`.

In [11]:
class Animal:
    def hablar(self):
        raise NotImplementedError

class Perro(Animal):
    def hablar(self):
        return "guau"

class Gato(Animal):
    def hablar(self):
        return "miau"

def coro(animales):
    return [a.hablar() for a in animales]

print(coro([Perro(), Gato(), Perro()]))

['guau', 'miau', 'guau']


## Trampas comunes y buenas prácticas

- Usa **composición** por defecto; hereda cuando “es un” sea realmente claro.
- Evita jerarquías profundas: si te cuesta explicar la relación, probablemente no es herencia.
- Herencia múltiple: posible, pero compleja. **No recomendada** salvo casos controlados (mixins pequeños).
- Polimorfismo en Python suele ser *duck typing*: diseña por interfaz (métodos esperados), no por tipo.

## Mini-práctica

1) Composición:
   - Crea `Pedido` que “tenga” una lista de `LineaPedido(sku, qty, price)` y un método `total()`.

2) Herencia:
   - Crea `Vehiculo` y `Bicicleta(Vehiculo)` con un método extra `timbre()`.

3) Polimorfismo:
   - Crea dos clases con método `exportar()` (por ejemplo `ExportarCSV`, `ExportarJSON`) y una función que llame a `exportar()` sin saber el tipo.